# Adaptive Evidence-Aware RAG Comparison Demo

This notebook compares **Standard Dense RAG** and **Adaptive Evidence-Aware RAG** side-by-side using a corpus containing duplicate documents. It records how the adaptive components (Independence & Utility scorer with dynamic NLI entailment logic) successfully deduplicate identical context information to prevent wasting token space and guide clean generator reasoning.

In [1]:
import os
import sys
from pathlib import Path
import contextlib

# ==================================================================
# CRITICAL JUPYTER CRASH PREVENTION (Must be run before any imports)
# ==================================================================
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# Disable tqdm notebook widgets which frequently crash VS Code Jupyter
import tqdm
import tqdm.auto
tqdm.auto.tqdm = tqdm.std.tqdm
tqdm.tqdm = tqdm.std.tqdm

# Monkey-patch Hugging Face lock to avoid deadlock issues on Windows
import huggingface_hub.utils
import huggingface_hub.file_download
@contextlib.contextmanager
def dummy_weak_file_lock(*args, **kwargs):
    yield None

huggingface_hub.utils.WeakFileLock = dummy_weak_file_lock
huggingface_hub.file_download.WeakFileLock = dummy_weak_file_lock

# Ensure project root is in python path
sys.path.append(str(Path(os.getcwd()).parent))

# ==================================================================
# PIPELINE EXECUTION
# ==================================================================
from src.retriever import EvidenceAwareRetriever
from src.utils import load_config
from src.cli import _demo_corpus

print("Environment configured successfully. Loading models...")

# Load compare config
config_path = "../configs/compare_config.yaml"
config = load_config(config_path)

retriever = EvidenceAwareRetriever(
    embedder_name=config["models"]["embedder"]["name"],
    reranker_name=config["models"]["reranker"]["name"],
    nli_model_name=config["models"]["nli"]["name"],
    independence_config=config.get("independence", {}),
    utility_config=config.get("utility", {}),
    search_policy_config=config.get("search_policy", {}),
    stability_config=config.get("stability", {}),
    top_k=config["retrieval"]["top_k"],
    use_independence=True,
    use_utility=True,
    use_search_policy=False,
    use_stability=False,
)
print("Models loaded successfully!")

# Index corpus
docs = _demo_corpus()
print(f"\nIndexing demo corpus ({len(docs)} duplicated documents)...")
retriever.index_documents(docs)

# Execute comparison query
question = "Who invented the transformer architecture?"
print(f"\nRunning query: '{question}'\n")
result = retriever.run_pipeline(question)

print("=" * 75)
print("COMPARISON SUMMARY")
print("=" * 75)

print("\n[1] STANDARD DENSE RAG (Basic Retrieval)")
print(f"Total documents retrieved: {len(result.original_documents)}")
unique_std = len(set(result.original_documents))
print(f"Unique documents: {unique_std} / {len(result.original_documents)} ({(100 * unique_std / len(result.original_documents)):.1f}% unique)")
print("-" * 50)
for i, doc in enumerate(result.original_documents[:5], 1):
    print(f"  Doc {i}: {doc[:100]}...")

print("\n[2] OUR ADAPTIVE EVIDENCE-AWARE RAG (With Independence & Utility Filtering)")
print(f"Total documents returned : {len(result.filtered_documents)}")
unique_ours = len(set(result.filtered_documents))
print(f"Unique documents: {unique_ours} / {len(result.filtered_documents)} ({(100 * unique_ours / max(1, len(result.filtered_documents))):.1f}% unique)")
print(f"Independence score: {result.independence_score:.4f}")
print(f"Utility score     : {result.utility_score:.4f}")
print(f"Overall quality   : {result.overall_quality:.4f}")
print("-" * 50)
for i, doc in enumerate(result.filtered_documents[:5], 1):
    print(f"  Doc {i}: {doc[:100]}...")


Environment configured successfully. Loading models...
[Retriever] Loading embedder: BAAI/bge-small-en-v1.5


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:  81%|████████▏ | 162/199 [00:00<00:00, 1619.43it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1591.77it/s]

[Retriever] Initializing Independence Scorer
[IndependenceScorer] Loading embedder: BAAI/bge-small-en-v1.5


...

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 1907.57it/s]

Models loaded successfully!

Indexing demo corpus (1000 duplicated documents)...
[Retriever] Indexing 1000 documents


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

[Retriever] Indexed documents with embedding shape: (1000, 384)

Running query: 'Who invented the transformer architecture?'



COMPARISON SUMMARY

[1] STANDARD DENSE RAG (Basic Retrieval)
Total documents retrieved: 10
Unique documents: 1 / 10 (10.0% unique)
--------------------------------------------------
  Doc 1: The transformer architecture was introduced in 2017 by Vaswani et al. in the paper 'Attention Is All...
  Doc 2: The transformer architecture was introduced in 2017 by Vaswani et al. in the paper 'Attention Is All...
  Doc 3: The transformer architecture was introduced in 2017 by Vaswani et al. in the paper 'Attention Is All...
  Doc 4: The transformer architecture was introduced in 2017 by Vaswani et al. in the paper 'Attention Is All...
  Doc 5: The transformer architecture was introduced in 2017 by Vaswani et al. in the paper 'Attention Is All...

[2] OUR ADAPTIVE EVIDENCE-AWARE RAG (With Independence & Utility Filtering)
Total documents returned : 1
Unique documents: 1 / 1 (100.0% unique)
Independence score: 0.1000
Utility score     : 0.6000
Overall quality   : 0.3950
--------------------------

## 5. Generator Model Comparison

Now that we have filtered down the context to a single highly useful and independent document, let's see how different LLMs synthesize it.
We will compare:
1. **Baseline Ollama Model:** Standard `phi3` or `llama3` running locally.
2. **Base Pre-trained Model:** `Qwen/Qwen1.5-0.5B-Chat` without any fine-tuning.
3. **Our Custom Fine-Tuned Model:** `Qwen1.5-0.5B-Evidence-RAG` (using our LoRA adapters).

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

question = "Who invented the transformer architecture?"
docs = result.filtered_documents
context = "\n\n".join([f"[Doc {i+1}] {doc}" for i, doc in enumerate(docs)])

prompt = f'''You are an advanced Evidence-Aware AI Assistant. \n"
Answer the following user question using ONLY the provided verified evidence.\n"
If the evidence does not contain the answer, say "I don't have enough verified evidence to answer this."\n\n"
Evidence:\n"
{context}\n\n"
Question:\n"
{question}\n\n"
Answer:'''

print("========== GENERATOR COMPARISON ==========")


In [ ]:
# 1. Baseline Ollama
try:
    ollama = ChatOllama(model="phi3", temperature=0.0)
    ollama_res = ollama.invoke([HumanMessage(content=prompt)])
    print(f"\n[1] Baseline Ollama (phi3):\n{ollama_res.content}")
except Exception as e:
    print(f"\n[1] Baseline Ollama (phi3):\nError or not running - {e}")


In [ ]:
# 2. Base Pre-trained Model (Qwen-0.5B)
try:
    model_name = "Qwen/Qwen1.5-0.5B-Chat"
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    base_model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.float16)
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
    outputs = base_model.generate(**inputs, max_new_tokens=100, temperature=0.1, do_sample=True)
    base_res = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"\n[2] Base Model (Qwen-0.5B):\n{base_res}")
except Exception as e:
    print(f"\n[2] Base Model (Qwen-0.5B):\nError loading base model - {e}")


In [ ]:
# 3. Our Fine-Tuned Model (Qwen-0.5B + LoRA)
lora_path = "../models/qwen-evidence-rag-lora-final"
try:
    finetuned_model = PeftModel.from_pretrained(base_model, lora_path)
    outputs_ft = finetuned_model.generate(**inputs, max_new_tokens=100, temperature=0.1, do_sample=True)
    ft_res = tokenizer.decode(outputs_ft[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"\n[3] Our Fine-Tuned Model:\n{ft_res}")
except Exception as e:
    print(f"\n[3] Our Fine-Tuned Model:\nNot trained yet! Run finetune_synthesizer.ipynb first. Error: {e}")
